# Nested feature selection — SVM (polynomial kernel)

Leakage-free feature selection for **svm_poly** on the 3-class ground-type problem. Forward selection
runs **inside each outer leave-user-out fold**, choosing features on an inner grouped CV of only that
fold's training users — so the reported macro-F1 has no selection bias (the honest 'what generalises'
number). Reports per-classifier: honest grouped macro-F1, and which features get selected across folds.

Poly kernels amplify large-magnitude features; slowest model -> SUBSAMPLE + max_iter essential.

**Thermal/cost guards** (kernel SVMs are O(n^2-3)): `SUBSAMPLE_PER_CLASS` caps train-fold size,
`SVM_MAX_ITER` stops a non-converging fit instead of hanging, `SFS_JOBS=4` uses half the cores.
Every fold prints live progress so the cell never looks frozen.

*Honest limit:* 3-class CV rests on 3 users (loose caps it) -> large fold-to-fold variance; treat
gaps under ~3 macro-F1 points as noise. Classifier is UNTUNED (default hyperparameters).

In [1]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from sklearn.svm import SVC

# ================= CONFIG =================
DATA_DIR='..'
MAX_K=15                       # SFS cap (selection plateaus early; keeps runtime bounded)
SFS_TOL=0.0005                  # stop adding features when inner gain < this
SVM_MAX_ITER=50000             # safety cap so a non-converging SVM STOPS, not hangs
SUBSAMPLE_PER_CLASS=2500   # cap per-class train size (kernel SVM cost/heat); None=all
SFS_JOBS=4                     # parallel feature-trials across cores (thermal-safe: half of 8)
CLASS_WEIGHT_STRENGTH=0.5
RANDOM_STATE=42

DROP={'total_p_mean','total_pmax','impact_mag','impact_mag_norm','heel_impulse','heel_mean_ratio',
      'fore_mean_ratio','heel_pmax_ratio','fore_pmax_ratio'}|{f'pressure_{i:02d}_mean_ratio' for i in range(1,13)}
META={'foot','step_id','surface','source_user','source_recording'}

dfs=[]
for fn,lab in [('grass_dry.csv','grass'),('loose_dry.csv','loose'),('paved_dry.csv','paved')]:
    d=pd.read_csv(f'{DATA_DIR}/{fn}'); d['surface']=lab; dfs.append(d)
df=pd.concat(dfs,ignore_index=True)
FEATS=[c for c in df.columns if c not in META and c not in DROP]
classes=sorted(df.surface.unique()); y=df.surface.values; grp=df.source_user.values
_users_per_class=df.groupby('surface')['source_user'].nunique()
N_SPLITS=int(_users_per_class.min())
print(f'{len(df)} steps | {len(FEATS)} features | users/class {_users_per_class.to_dict()} -> N_SPLITS={N_SPLITS}')
print(f'classifier: svm_poly | SUBSAMPLE_PER_CLASS={SUBSAMPLE_PER_CLASS} | SFS_JOBS={SFS_JOBS}')

15664 steps | 49 features | users/class {'grass': 4, 'loose': 3, 'paved': 4} -> N_SPLITS=3
classifier: svm_poly | SUBSAMPLE_PER_CLASS=2500 | SFS_JOBS=4


In [2]:
def _cw_from(ytr):
    if not True: return None
    cs,cnt=np.unique(ytr,return_counts=True); n,k=len(ytr),len(cs)
    return {c:(n/(k*ct))**CLASS_WEIGHT_STRENGTH for c,ct in zip(cs,cnt)}

def make_pipe(ytr):
    _cw=_cw_from(ytr)
    return Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='poly', degree=3, gamma='scale', C=1.0, class_weight=_cw, max_iter=SVM_MAX_ITER, random_state=42))])

def _subsample(idx, yv, per_class, seed=RANDOM_STATE):
    if not per_class: return idx
    rng=np.random.default_rng(seed); keep=[]
    for c in np.unique(yv[idx]):
        ci=idx[yv[idx]==c]; keep+=list(rng.choice(ci,min(len(ci),per_class),replace=False))
    return np.array(sorted(keep))

Xall=df[FEATS].fillna(0).values

def fit_score(cols, tr, te, yv):
    tr=_subsample(tr, yv, SUBSAMPLE_PER_CLASS)
    p=make_pipe(yv[tr]); p.fit(Xall[np.ix_(tr,cols)], yv[tr])
    return f1_score(yv[te], p.predict(Xall[np.ix_(te,cols)]), labels=classes, average='macro')

def inner_cv_score(cols, tr, yv, gv):
    inner=StratifiedGroupKFold(n_splits=max(2,min(N_SPLITS-1,len(np.unique(gv)))),shuffle=True,random_state=0)
    sc=[]
    for itr,ite in inner.split(Xall[tr][:,cols], yv[tr], gv):
        if len(np.unique(yv[tr][itr]))<len(classes): continue
        sub_tr=tr[itr]; sub_te=tr[ite]
        sc.append(fit_score(cols, sub_tr, sub_te, yv))
    return np.mean(sc) if sc else -1

In [3]:
# ===== nested forward selection with LIVE progress =====
splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
_folds=[(tr,te) for tr,te in splitter.split(Xall,y,grp) if len(np.unique(y[tr]))==len(classes)]
print(f'nested forward selection for svm_poly: {len(_folds)} outer folds, MAX_K={MAX_K}')
outer_mf1=[]; sel_counts={}; per_fold_sets=[]
for _fi,(tr,te) in enumerate(_folds,1):
    t0=time.time(); held=np.unique(grp[te])
    print(f'  fold {_fi}/{len(_folds)} (hold out {held}) — forward-selecting...', flush=True)
    yv,gv=y, grp[tr]
    chosen=[]; remaining=list(range(len(FEATS))); best=-1; step=0
    while remaining and len(chosen)<MAX_K:
        scores=Parallel(n_jobs=SFS_JOBS,prefer='processes')(
            delayed(inner_cv_score)(chosen+[c], tr, yv, gv) for c in remaining)
        s_best,c_best=max(zip(scores,remaining))
        if s_best-best<SFS_TOL: break
        chosen.append(c_best); best=s_best; remaining.remove(c_best); step+=1
        print(f'      + {FEATS[c_best]}  (inner MacroF1 {best*100:.1f}%, {step} feats, {time.time()-t0:.0f}s)', flush=True)
    pf=fit_score(chosen, tr, te, y)
    outer_mf1.append(pf); per_fold_sets.append([FEATS[c] for c in chosen])
    for c in chosen: sel_counts[FEATS[c]]=sel_counts.get(FEATS[c],0)+1
    print(f'    -> fold done: {len(chosen)} feats, held-out MacroF1 {pf*100:.1f}%  ({time.time()-t0:.0f}s)\n', flush=True)
outer_mf1=np.array(outer_mf1)
print(f'=== svm_poly: nested (leakage-free) grouped MacroF1 {outer_mf1.mean()*100:.1f}% '
      f'(sd {outer_mf1.std()*100:.1f}) | per-fold {np.round(outer_mf1*100,1)} ===')

nested forward selection for svm_poly: 3 outer folds, MAX_K=15
  fold 1/3 (hold out ['Sam']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 40.2%, 1 feats, 16s)
      + gyro_x_std  (inner MacroF1 48.5%, 2 feats, 29s)
      + gu_press_jitter  (inner MacroF1 50.6%, 3 feats, 39s)
      + stance_duration_roll3_std  (inner MacroF1 51.7%, 4 feats, 48s)
      + heel_ripple_frac  (inner MacroF1 51.9%, 5 feats, 57s)
      + heel_skew  (inner MacroF1 52.1%, 6 feats, 66s)
      + fall_slope  (inner MacroF1 53.3%, 7 feats, 76s)
    -> fold done: 7 feats, held-out MacroF1 40.7%  (86s)

  fold 2/3 (hold out ['Kristian']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 41.0%, 1 feats, 15s)
      + gu_load_first_frac  (inner MacroF1 46.3%, 2 feats, 27s)
      + gyro_x_std  (inner MacroF1 47.2%, 3 feats, 38s)
      + gyro_x_strike  (inner MacroF1 48.0%, 4 feats, 50s)
    -> fold done: 4 feats, held-out MacroF1 47.5%  (61s)

  fold 3/3 (hold out ['Catherine

In [4]:
# ===== selection frequency: which features this classifier relies on across folds =====
freq=sorted(sel_counts.items(), key=lambda kv:-kv[1])
print(f'Feature selection frequency for svm_poly (across {len(_folds)} folds):')
for f,c in freq: print(f'  {c}/{len(_folds)}  {f}')
print('\nPer-fold selected sets:')
for i,s in enumerate(per_fold_sets,1): print(f'  fold {i}: {s}')
# a 'recommended set' = features chosen in a MAJORITY of folds (stable across users)
maj=[f for f,c in freq if c> len(_folds)//2]
print(f'\nStable set (selected in > half of folds): {maj}')
print('NOTE: with 3 users, per-fold sets vary; the STABLE set is the trustworthy takeaway, '
      'not any single fold\'s pick. Report the nested MacroF1 as this classifier\'s honest accuracy.')

Feature selection frequency for svm_poly (across 3 folds):
  3/3  orient_rest_a_roll3_std
  2/3  gyro_x_std
  2/3  heel_ripple_frac
  1/3  gu_press_jitter
  1/3  stance_duration_roll3_std
  1/3  heel_skew
  1/3  fall_slope
  1/3  gu_load_first_frac
  1/3  gyro_x_strike
  1/3  accel_jerk_ratio
  1/3  orient_range_a
  1/3  impact_mag_roll3_std
  1/3  loading_rate_norm
  1/3  gyro_y_strike
  1/3  rise_slope
  1/3  orient_rest_b_roll3_std
  1/3  gu_stance_frac

Per-fold selected sets:
  fold 1: ['orient_rest_a_roll3_std', 'gyro_x_std', 'gu_press_jitter', 'stance_duration_roll3_std', 'heel_ripple_frac', 'heel_skew', 'fall_slope']
  fold 2: ['orient_rest_a_roll3_std', 'gu_load_first_frac', 'gyro_x_std', 'gyro_x_strike']
  fold 3: ['orient_rest_a_roll3_std', 'accel_jerk_ratio', 'orient_range_a', 'heel_ripple_frac', 'impact_mag_roll3_std', 'loading_rate_norm', 'gyro_y_strike', 'rise_slope', 'orient_rest_b_roll3_std', 'gu_stance_frac']

Stable set (selected in > half of folds): ['orient_rest_a_